# 01 — AAA PDF parsing and cleaning

Rebuilds the Abdominal Aortic Aneurysm (AAA) preprocessing pipeline from the original PDFs.

Rules:
- original PDFs are never modified or deleted
- source text is not paraphrased
- metadata, grades, evidence levels, and URLs are stored only when the source supports them
- page numbers are 1-based
- every page remains traceable as `document_id + source_file + page_number`


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd.parent]:
    if (candidate / "ingestion" / "preprocess.py").exists():
        sys.path.insert(0, str(candidate))
        break

import ingestion.preprocess as cp

PROJECT_ROOT = cp.find_project_root()
print("clinical_preprocess loaded from:", cp.__file__)
print("Project root:", PROJECT_ROOT)
print("pdfplumber available:", cp.HAS_PDFPLUMBER)
print("OCR available:", cp.HAS_OCR)


clinical_preprocess loaded from: C:\Users\DELL\Downloads\aaa-clinical-rag\notebooks\clinical_preprocess.py
Project root: C:\Users\DELL\Downloads\aaa-clinical-rag
pdfplumber available: True
OCR available: False


## Discover PDFs and previous extraction bugs

Known bugs from earlier runs that this rebuild must not repeat:

1. Hyphen + newline joining turned `0.98-\n1.00` into `0.981.00`.
2. NICE `x.x.x` IDs were assigned from DOI fragments such as `2017.10.044`.
3. USPSTF `publication_year` was incorrectly set to 2014 (the prior recommendation being updated).
4. NICE contents pages with dotted leaders were treated as corrupted.
5. SVS title/divider slides need `low_text` (or similar) rather than invented OCR text.
6. Spaced output names such as `pages .json` must not be created.


In [2]:
pdfs = cp.discover_pdfs(PROJECT_ROOT)
print(f"Discovered {len(pdfs)} PDF(s)\n")
for pdf in pdfs:
    identity = cp.identify_document(pdf, PROJECT_ROOT)
    assessment = cp.assess_pdf_validity(pdf)
    meta = cp.pdf_metadata(pdf)
    print(pdf.name)
    print("  path:", cp.relative_posix(pdf, PROJECT_ROOT))
    print("  pages:", meta.get("page_count"))
    print("  document_id:", identity["document_id"])
    print("  identification:", identity["identification_method"])
    print("  valid:", assessment["valid"], assessment.get("reason"))
    print()


Discovered 4 PDF(s)



abdom-aortic-aneurysm-screening-final-rs.pdf
  path: data/pdfs/abdom-aortic-aneurysm-screening-final-rs.pdf
  pages: 8
  document_id: USPSTF_2019
  identification: page1_content
  valid: True None

abdominal-aortic-aneurysm-diagnosis-and-management-pdf-66141843642565.pdf
  path: data/pdfs/abdominal-aortic-aneurysm-diagnosis-and-management-pdf-66141843642565.pdf
  pages: 53
  document_id: NICE_NG156
  identification: page1_content
  valid: True None

ESVS_2024_AAA_Guidelines.pdf
  path: data/pdfs/ESVS_2024_AAA_Guidelines.pdf
  pages: 140
  document_id: ESVS_2024
  identification: filename
  valid: True None

SVS_Guideline_AAA_Slides_0.pdf
  path: data/pdfs/SVS_Guideline_AAA_Slides_0.pdf
  pages: 48
  document_id: SVS_2018
  identification: filename
  valid: True None



In [3]:
print("Hyphenation / numeric-range unit tests")
hyphen_results = cp.hyphenation_unit_tests()
display(pd.DataFrame(hyphen_results)[["name", "raw", "clean", "pass"]])
assert all(row["pass"] for row in hyphen_results), "Hyphenation tests failed"
print("All hyphenation tests passed.")


Hyphenation / numeric-range unit tests


,name,raw,clean,pass
0,letter hyphenation joined,coun-\ntries,countries,True
1,decimal range preserved,0.98-\n1.00,0.98-\n1.00,True
2,en-dash range preserved,0.98–1.00,0.98–1.00,True
3,page range preserved,301-304,301-304,True
4,measurement preserved,5.5 cm,5.5 cm,True
5,age range preserved,65-75 years,65-75 years,True


All hyphenation tests passed.


## Run extraction, cleaning, recommendations, and metadata


In [4]:
result = cp.run_pipeline(PROJECT_ROOT)
cp.print_preprocessing_summary(result["report"])
print("\nDeleted obsolete generated files:")
for item in result.get("deleted_files") or []:
    print(" -", item)


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


DOCUMENTS
- discovered: 4
- processed: 4
- failed: 0
- excluded: 0

PAGES
- total: 249
- ok: 243
- low_text: 5
- image_only: 0
- OCR required: 0
- corrupted: 1

NUMERIC PRESERVATION
- raw numeric tokens: 14788
- clean numeric tokens: 14658
- missing tokens: 130
- loss ratio: 0.008790911549905328
- critical losses: []

RECOMMENDATIONS
- total: 233
- with grades: 4
- with evidence levels: 104
- with source excerpts: 233
- traceability: PASS

HYPHENATION TESTS
- letter hyphenation joined: PASS
- decimal range preserved: PASS
- en-dash range preserved: PASS
- page range preserved: PASS
- measurement preserved: PASS
- age range preserved: PASS

STATUS: PASS

Deleted obsolete generated files:
 - data/processed/document_metadata.json
 - data/processed/extraction_report.json
 - data/processed/pages.json
 - data/processed/pages_df.parquet
 - data/processed/recommendations.json
 - data/chunks/chunks.json
 - data/embeddings/embedded_chunks.json
 - data/embeddings/embeddings.npy
 - data/embeddings

In [5]:
docs = pd.DataFrame(json.loads((PROJECT_ROOT / "data" / "processed" / "document_metadata.json").read_text(encoding="utf-8")))
pages = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "pages_df.parquet")
recs = pd.DataFrame(json.loads((PROJECT_ROOT / "data" / "processed" / "recommendations.json").read_text(encoding="utf-8")))
report = json.loads((PROJECT_ROOT / "data" / "processed" / "extraction_report.json").read_text(encoding="utf-8"))

print("Documents")
display(docs[["document_id", "document_name", "document_type", "source_type", "source_organization", "publication_year", "page_count", "extraction_status", "source_url"]])
print("\nPage extraction status")
display(pages.groupby(["document_id", "extraction_status"]).size().unstack(fill_value=0))
print("\nRecommendations by document")
if recs.empty:
    print("No recommendations extracted.")
else:
    display(recs.groupby("document_id").size().rename("n_recommendations").to_frame())


Documents


,document_id,document_name,document_type,source_type,source_organization,publication_year,page_count,extraction_status,source_url
0,USPSTF_2019,Screening for Abdominal Aortic Aneurysm: US Preventive Services Task Force Recommendation Statement,government/public health recommendation,government/public health recommendation,US Preventive Services Task Force,2019,8,ok,doi:10.1001/jama.2019.18928
1,NICE_NG156,Abdominal aortic aneurysm: diagnosis and management,official guideline,official guideline,National Institute for Health and Care Excellence (NICE),2020,53,ok,www.nice.org.uk/guidance/ng156
2,ESVS_2024,Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice Guidelines on the Management ...,official guideline,official guideline,European Society for Vascular Surgery (ESVS),2024,140,corrupted,https://doi.org/10.1016/j.ejvs.2023.11.002
3,SVS_2018,Care of Patients with an Abdominal Aortic Aneurysm,official guideline,official guideline,Society for Vascular Surgery,2018,48,partial,vsweb.org/Guidelines



Page extraction status


extraction_status,corrupted,low_text,ok
document_id,,,
ESVS_2024,1,0,139
NICE_NG156,0,0,53
SVS_2018,0,5,43
USPSTF_2019,0,0,8



Recommendations by document


,n_recommendations
document_id,
ESVS_2024,60
NICE_NG156,57
SVS_2018,110
USPSTF_2019,6


In [6]:
print("Sample pages (first row per document) - metadata AND real extracted text")
sample = pages.sort_values(["document_id", "page_number"]).groupby("document_id", as_index=False).head(1)
display(sample[["document_id", "page_number", "section_title", "section_source", "extraction_status", "character_count", "extraction_library"]])

# Metadata alone cannot show whether extraction worked. Print the actual text.
for _, row in sample.iterrows():
    text = str(row["clean_text"] or "")
    print("=" * 88)
    print(f"document : {row['document_id']}  ({row['source_file']})")
    print(f"page     : {row['page_number']}")
    print(f"section  : {row['section_title'] if pd.notna(row['section_title']) else '<none>'} [{row['section_source']}]")
    print(f"status   : {row['extraction_status']}   chars: {row['character_count']}   words: {row['word_count']}")
    print("-" * 88)
    print(text[:900] + (" ..." if len(text) > 900 else ""))

print("\nRecommendation sample")
if not recs.empty:
    cols = [c for c in ["recommendation_id", "document_id", "page_number", "section_title", "recommendation_grade", "evidence_level", "extraction_confidence"] if c in recs.columns]
    display(recs[cols].head(12))

processed = PROJECT_ROOT / "data" / "processed"
print("\nProcessed output files")
for path in sorted(processed.iterdir()):
    if path.is_file():
        print(f" - {path.name} ({path.stat().st_size} bytes)")

assert report["status"] == "PASS", report.get("errors")
assert set(p.name for p in processed.iterdir() if p.is_file()) == cp.PROCESSED_OUTPUT_NAMES
print("\nNotebook 01 outputs validated.")


Sample pages (first row per document) - metadata AND real extracted text


,document_id,page_number,section_title,section_source,extraction_status,character_count,extraction_library
0,ESVS_2024,1,CLINICAL PRACTICE GUIDELINE DOCUMENT,detected,ok,5859,pymupdf
140,NICE_NG156,1,Abdominal aortic aneurysm: diagnosis and management,detected,ok,248,pymupdf
193,SVS_2018,1,Care of Patients with an,detected,ok,134,pymupdf
241,USPSTF_2019,1,Screening for Abdominal Aortic Aneurysm,detected,ok,3204,pymupdf


document : ESVS_2024  (ESVS_2024_AAA_Guidelines.pdf)
page     : 1
section  : CLINICAL PRACTICE GUIDELINE DOCUMENT [detected]
status   : ok   chars: 5859   words: 795
----------------------------------------------------------------------------------------
CLINICAL PRACTICE GUIDELINE DOCUMENT
Editor’s Choice – European Society for Vascular Surgery (ESVS) 2024 Clinical
Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery
Aneurysms5
Anders Wanhainen a,*, Isabelle Van Herzeele a, Frederico Bastos Goncalves a, Sergi Bellmunt Montoya a, Xavier Berard a, Jonathan R. Boyle a,
Mario D’Oria a, Carlota F. Prendes a, Christos D. Karkos a, Arkadiusz Kazimierczak a, Mark J.W. Koelemay a, Tilo Kölbel a, Kevin Mani a,
Germano Melissano a, Janet T. Powell a, Santi Trimarchi a, Nikolaos Tsilimparis a
ESVS Guidelines Committee b, George A. Antoniou, Martin Björck, Raphael Coscas, Nuno V. Dias, Philippe Kolh, Sandro Lepidi, Barend M.E. Mees,
Timothy A. Resch, Jean Baptiste Ricco, Riikka Tu

,recommendation_id,document_id,page_number,section_title,recommendation_grade,evidence_level,extraction_confidence
0,NaN,USPSTF_2019,1,Screening for Abdominal Aortic Aneurysm,NaN,NaN,medium
1,NaN,USPSTF_2019,1,Screening for Abdominal Aortic Aneurysm,B recommendation,NaN,high
2,NaN,USPSTF_2019,1,Screening for Abdominal Aortic Aneurysm,C recommendation,NaN,high
3,NaN,USPSTF_2019,2,Summary of Recommendations,I statement,NaN,high
4,NaN,USPSTF_2019,2,Summary of Recommendations,NaN,NaN,medium
5,NaN,USPSTF_2019,2,Summary of Recommendations,B recommendation,NaN,high
6,1.1.1,NICE_NG156,6,Recommendations,NaN,NaN,high
7,1.1.2,NICE_NG156,6,Recommendations,NaN,NaN,high
8,1.1.3,NICE_NG156,7,Recommendations,NaN,NaN,high
9,1.1.4,NICE_NG156,7,Recommendations,NaN,NaN,high



Processed output files
 - document_metadata.json (4168 bytes)
 - extraction_report.json (2122 bytes)
 - pages.json (2373714 bytes)
 - pages_df.parquet (1116147 bytes)
 - recommendations.json (212369 bytes)

Notebook 01 outputs validated.
